# BCI / motor imagery / Phd project

In [1]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path 
from scipy.io import loadmat
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline 

In [2]:
data_folder = Path(r'D:\Coding\BCI project\data')
subjects = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']
ch_renamed = {
    'EEG-Fz':'Fz',
    'EEG-0':'FC3',
    'EEG-1':'FC1',
    'EEG-2':'FCz',
    'EEG-3':'FC2',
    'EEG-4':'FC4',
    'EEG-5':'C5',
    'EEG-C3':'C3',
    'EEG-6':'C1',
    'EEG-Cz':'Cz',
    'EEG-7':'C2',
    'EEG-C4':'C4',
    'EEG-8':'C6',
    'EEG-9':'CP3',
    'EEG-10':'CP1',
    'EEG-11':'CPz',
    'EEG-12':'CP2',
    'EEG-13':'CP4',
    'EEG-14':'P1',
    'EEG-Pz':'Pz',
    'EEG-15':'P2',
    'EEG-16':'POz',
}

In [3]:
def preprocessing(raw, ch_names):
    raw.rename_channels(ch_names)
    raw.set_channel_types({
        'EOG-left':'eog',
        'EOG-central':'eog',
        'EOG-right':'eog'
    })
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing = 'ignore')
    raw.annotations.rename = {'1023':'BAD_1023'}
    raw.filter(l_freq = 1, h_freq = None, fir_design='firwin')
    return raw

In [4]:
def ica_eog(raw):
    ica = mne.preprocessing.ICA(n_components = 20, max_iter = 'auto', random_state = 0)
    ica.fit(raw, reject_by_annotation = True)
    eog_indices, eog_scores = ica.find_bads_eog(raw, threshold = 3.0)
    ica.exclude = eog_indices
    ica.apply(raw)
    return raw

In [5]:
def load_test_labels(subject):
    mat_files = loadmat(Path(r'D:\Coding\BCI project\data/true labels') / f'{subject}E.mat')
    all_test_labels = mat_files['classlabel'].flatten()
    mask = (all_test_labels == 1) | (all_test_labels == 2)
    labels_test = all_test_labels[mask] - 1
    return labels_test, mask

In [6]:
def epoching_train(raw):
    raw.set_eeg_reference(projection = True)
    raw.filter(l_freq=8, h_freq=30, fir_design='firwin')
    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, 
                        event_id = {'left hand':event_id['769'], 'right hand':event_id['770']}, 
                        tmin = -1, tmax = 4, proj = True, picks = 'eeg', baseline = None, 
                        preload = True)
    epochs_croped = epochs.copy().crop(tmin = 1.0, tmax = 2)
    epochs_data = epochs.get_data(copy = False)
    epochs_croped_data = epochs_croped.get_data(copy = False)
    labels = epochs.events[:, -1] - event_id['769']
    return epochs, epochs_croped, epochs_data, epochs_croped_data, labels

In [7]:
def epoching_test(raw):
    raw.set_eeg_reference(projection = True)
    raw.filter(l_freq=8, h_freq=30, fir_design='firwin')
    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id = {'unknown':event_id['783']}, tmin = -1,
                        tmax = 4, proj = True, picks = 'eeg', baseline = None, preload = True)
    epochs_croped = epochs.copy().crop(tmin = 1.0, tmax = 2)
    epochs_data = epochs.get_data(copy = False)
    epochs_croped_data = epochs_croped.get_data(copy = False)
    return epochs, epochs_croped, epochs_data, epochs_croped_data

In [8]:
results = {}
class_balance = []

for subject in subjects:
    
    train_files = data_folder / f'{subject}T.gdf' 
    test_files = data_folder / f'{subject}E.gdf'
    raw_train = mne.io.read_raw_gdf(train_files, preload = True)
    raw_test = mne.io.read_raw_gdf(test_files,  preload = True)

    raw_train = preprocessing(raw_train, ch_renamed)
    raw_test = preprocessing(raw_test, ch_renamed)

    raw_train = ica_eog(raw_train)
    raw_test = ica_eog(raw_test)

    labels_test, mask = load_test_labels(subject)

    epochs_train, epochs_train_croped, epochs_train_data, epochs_train_croped_data, labels_train = epoching_train(raw_train)
    epochs_test, epochs_test_croped, epochs_test_data, epochs_test_croped_data = epoching_test(raw_test)
    epochs_test_croped_data = epochs_test_croped_data[mask]

    csp = CSP(n_components = 4, reg = None, log = True, norm_trace = False)
    lda = LinearDiscriminantAnalysis()
    classification = Pipeline([('CSP', csp), ('LDA', lda)])
    classification.fit(epochs_train_croped_data, labels_train)
    score = classification.score(epochs_test_croped_data, labels_test)
    results[subject] = score

    class_balance.append(np.mean(labels_test == labels_test[0]))

final_score = np.mean(list(results.values()))
final_std = np.std(list(results.values()))
class_balance_mean = np.mean(class_balance)
chance_level = max(class_balance_mean, 1 - class_balance_mean)

print(f'Total classification precision is {final_score * 100}%')
print(f'Standard deviation is {final_std}')
print(f'Chance level is {chance_level * 100}%')
for sbjct, scr in results.items():
    print(f'subject {sbjct} - score {scr}')

Extracting GDF parameters from D:\Coding\BCI project\data\A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...


C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Reading 0 ... 672527  =      0.000 ...  2690.108 secs...
Extracting GDF parameters from D:\Coding\BCI project\data\A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 686999  =      0.000 ...  2747.996 secs...


C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 20

...

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 20

C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Extracting GDF parameters from D:\Coding\BCI project\data\A09E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 675097  =      0.000 ...  2700.388 secs...


C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 20